# Sleep staging on iSLEEPS

Data: `processed7` = one `.npz` per subject with
`x` of shape `[n_epochs, 7, 3000]` (30 s @ 100 Hz) and `y` in `{0..4}` = W, N1, N2, N3, REM.
Channels: `C4:M1, C3:M2, O2:M1, O1:M2, E1:M2, E2:M2, EMG`.

Four experiments, each trained here from scratch:

1. Engineered features + gradient boosting + HMM decoding
2. Raw-signal CNN, per-epoch
3. CNN + BiLSTM over epoch sequences
4. Hemispheric-asymmetry graph + state-space model

All use the same subject-independent folds, so the numbers are comparable.

##0. Setup

In [1]:
import os, glob, json, math, time
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.signal import welch, butter, filtfilt, hilbert
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

DATA = "/kaggle/input/datasets/abwahi/ischemic-sleep-stage/processed7"

FS          = 100          # sampling rate
N_CLASSES   = 5
CLASS_NAMES = ["W", "N1", "N2", "N3", "R"]
DUPLICATE   = {28}         # SN28 is a bit-identical copy of SN15

N_SUBJECTS  = 20           # None = all 99
N_FOLDS     = 3            # paper uses 10
EPOCHS      = 10
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42); np.random.seed(42)
print("data dir:", DATA, "| device:", DEVICE)

data dir: /kaggle/input/datasets/abwahi/ischemic-sleep-stage/processed7 | device: cuda


In [2]:
# list subjects, drop the duplicate recording
paths = sorted(glob.glob(os.path.join(DATA, "SN*.npz")))
sids  = [int(os.path.basename(p)[2:-4]) for p in paths]
sids  = [s for s in sids if s not in DUPLICATE]
if N_SUBJECTS: sids = sids[:N_SUBJECTS]
print(f"{len(sids)} subjects")

d = np.load(os.path.join(DATA, f"SN{sids[0]}.npz"), allow_pickle=True)
print("keys:", list(d.keys()))
print("x:", d["x"].shape, d["x"].dtype, "| y:", d["y"].shape)
print("channels:", list(d["channels"]))

20 subjects
keys: ['x', 'y', 'channels', 'subject', 'sfreq']
x: (717, 7, 3000) float16 | y: (717,)
channels: [np.str_('C4:M1'), np.str_('C3:M2'), np.str_('O2:M1'), np.str_('O1:M2'), np.str_('E1:M2'), np.str_('E2:M2'), np.str_('EMG')]


In [3]:
# load every subject into memory once (float32, ~30 MB per subject)
X, Yl = {}, {}
for s in sids:
    d = np.load(os.path.join(DATA, f"SN{s}.npz"), allow_pickle=True)
    X[s] = d["x"].astype(np.float32)
    Yl[s] = d["y"].astype(np.int64)
tot = sum(len(v) for v in Yl.values())
cnt = np.bincount(np.concatenate(list(Yl.values())), minlength=5)
print(f"{tot:,} epochs")
for c, n in zip(CLASS_NAMES, cnt):
    print(f"  {c:3s} {n:7,d}  {100*n/cnt.sum():5.1f}%")
print(f"\nN2:N3 imbalance = {cnt[2]/cnt[3]:.1f} : 1  -> accuracy alone will mislead")

18,621 epochs
  W     6,036   32.4%
  N1    1,576    8.5%
  N2    7,525   40.4%
  N3    1,646    8.8%
  R     1,838    9.9%

N2:N3 imbalance = 4.6 : 1  -> accuracy alone will mislead


In [4]:
# subject-independent folds: every epoch of a patient is entirely in train or test
def make_folds(subjects, n_splits, seed=42):
    rng = np.random.RandomState(seed)
    sh = list(subjects); rng.shuffle(sh)
    groups = [sh[i::n_splits] for i in range(n_splits)]
    return [(sorted(s for s in sh if s not in groups[k]), sorted(groups[k]))
            for k in range(n_splits)]

FOLDS = make_folds(sids, N_FOLDS)
for i, (tr, te) in enumerate(FOLDS):
    print(f"fold {i}: train {len(tr)} subjects | test {len(te)}")

fold 0: train 13 subjects | test 7
fold 1: train 13 subjects | test 7
fold 2: train 14 subjects | test 6


In [5]:
# shared metrics + HMM decoding (used by every experiment)
def metrics(y, p):
    return dict(acc   = accuracy_score(y, p),
                mf1   = f1_score(y, p, average="macro", zero_division=0),
                kappa = cohen_kappa_score(y, p),
                pcf   = f1_score(y, p, average=None, labels=range(5), zero_division=0))

def report(name, y, p):
    m = metrics(y, p)
    print(f"{name:32s} acc={m['acc']:.4f}  macroF1={m['mf1']:.4f}  kappa={m['kappa']:.4f}")
    print("   per-class F1: " + "  ".join(f"{c}={v:.3f}" for c, v in zip(CLASS_NAMES, m["pcf"])))
    return m

def fit_hmm(seqs):
    """transition matrix + initial prior from training hypnograms"""
    A = np.ones((5,5)); pi = np.ones(5)
    for y in seqs:
        pi[y[0]] += 1
        for a,b in zip(y[:-1], y[1:]): A[a,b] += 1
    A /= A.sum(1, keepdims=True); pi /= pi.sum()
    return np.log(A + 1e-12), np.log(pi + 1e-12)

def viterbi(logp, logA, logpi):
    """most likely stage sequence given per-epoch log-probabilities"""
    T = logp.shape[0]
    dp = np.zeros((T,5)); bp = np.zeros((T,5), int)
    dp[0] = logpi + logp[0]
    for t in range(1, T):
        sc = dp[t-1][:,None] + logA
        bp[t] = sc.argmax(0); dp[t] = sc.max(0) + logp[t]
    path = np.zeros(T, int); path[-1] = dp[-1].argmax()
    for t in range(T-2, -1, -1): path[t] = bp[t+1, path[t+1]]
    return path

RESULTS = {}

## Experiment 1: Engineered features + gradient boosting + HMM

Describe each 30 s epoch with the quantities a human scorer actually reads: band powers,
spectral entropy, Hjorth descriptors, and event features for spindles, slow waves, eye
movement and chin tone. Then a gradient-boosted tree per epoch, and HMM decoding across
the night.

In [6]:
BANDS = [("delta",0.5,4), ("theta",4,8), ("alpha",8,12), ("sigma",12,16), ("beta",16,30)]

def bandpass(x, lo, hi):
    b, a = butter(4, [lo/(FS/2), min(hi,49.0)/(FS/2)], btype="band")
    return filtfilt(b, a, x)

def epoch_features(ep):
    """ep [7,3000] -> 1-D feature vector"""
    f = []
    for c in range(7):
        x = ep[c].astype(np.float64)
        fr, P = welch(x, fs=FS, nperseg=256)
        tot = P[(fr>=0.5)&(fr<=30)].sum() + 1e-12
        for _, lo, hi in BANDS:                       # absolute + relative band power
            bp = P[(fr>=lo)&(fr<hi)].sum()
            f += [np.log(bp + 1e-12), bp/tot]
        pn = P/P.sum();  f.append(-(pn*np.log(pn+1e-12)).sum())        # spectral entropy
        cs = np.cumsum(P)/P.sum(); f.append(fr[np.searchsorted(cs,0.95)])  # spectral edge
        d1, d2 = np.diff(x), np.diff(x, 2)            # Hjorth
        v0 = x.var()+1e-12; v1 = d1.var()+1e-12; v2 = d2.var()+1e-12
        mob = np.sqrt(v1/v0)
        f += [np.log(v0), mob, np.sqrt(v2/v1)/(mob+1e-12)]
        f += [np.mean(np.abs(x)), np.percentile(np.abs(x),90),
              float(((x[:-1]*x[1:])<0).mean())]       # amplitude + zero-crossing rate
    # ---- event features -----------------------------------------------------
    for c in range(4):                                # EEG: spindles + slow waves
        env = np.abs(hilbert(bandpass(ep[c].astype(np.float64), 11, 16)))
        thr = np.quantile(env, 0.90)
        onsets = int(((env[1:]>thr) & (env[:-1]<=thr)).sum())
        f += [onsets, env.mean(), env.var()]
        sw = bandpass(ep[c].astype(np.float64), 0.5, 4)
        f += [sw.max()-sw.min(), np.mean(np.abs(sw))]
    for c in (4,5):                                   # EOG: ocular movement
        dx = np.diff(ep[c].astype(np.float64))
        f += [np.log(np.mean(dx**2)+1e-12), np.percentile(np.abs(dx),95)]
    emg = ep[6].astype(np.float64)                    # EMG: tone / atonia
    f += [np.log(np.sqrt(np.mean(emg**2))+1e-12), np.percentile(np.abs(emg),90)]
    return np.asarray(f, dtype=np.float32)

print("feature dim:", len(epoch_features(X[sids[0]][0])))

feature dim: 152


In [7]:
t0 = time.time()
F = {}
for i, s in enumerate(sids):
    F[s] = np.stack([epoch_features(e) for e in X[s]])
    if (i+1) % 5 == 0:
        print(f"  {i+1}/{len(sids)} subjects  ({time.time()-t0:.0f}s)", flush=True)
print(f"done in {(time.time()-t0)/60:.1f} min | feature matrix per subject: {F[sids[0]].shape}")

  5/20 subjects  (89s)
  10/20 subjects  (188s)
  15/20 subjects  (281s)
  20/20 subjects  (377s)
done in 6.3 min | feature matrix per subject: (717, 152)


In [8]:
def context_stack(Fs, k=3):
    """concatenate each epoch with its +-k neighbours (sleep is autocorrelated)"""
    Fp = np.pad(Fs, ((k,k),(0,0)), mode="edge")
    return np.concatenate([Fp[i:i+len(Fs)] for i in range(2*k+1)], axis=1)

def prep(s, k=3):
    z = (F[s] - F[s].mean(0)) / (F[s].std(0) + 1e-6)     # per-subject standardisation
    return context_stack(z, k)

print("context feature dim:", prep(sids[0]).shape[1])

context feature dim: 1064


In [9]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

yt_all, yp_raw, yp_hmm = [], [], []
for fi, (tr, te) in enumerate(FOLDS):
    Xtr = np.concatenate([prep(s) for s in tr])
    ytr = np.concatenate([Yl[s]  for s in tr])
    clf = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                        device=("cuda" if DEVICE=="cuda" else "cpu"),
                        n_jobs=-1, random_state=42)
    clf.fit(Xtr, ytr, sample_weight=compute_sample_weight("balanced", ytr))
    logA, logpi = fit_hmm([Yl[s] for s in tr])
    for s in te:
        prob = clf.predict_proba(prep(s))
        yt_all.append(Yl[s])
        yp_raw.append(prob.argmax(1))
        yp_hmm.append(viterbi(np.log(prob + 1e-12), logA, logpi))
    print(f"  fold {fi} done", flush=True)

yt = np.concatenate(yt_all)
RESULTS["1. features + XGBoost"]        = report("1. features + XGBoost", yt, np.concatenate(yp_raw))
RESULTS["1b. features + XGBoost + HMM"] = report("1b. + HMM decoding",    yt, np.concatenate(yp_hmm))

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [04:21:57] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  fold 0 done
  fold 1 done
  fold 2 done
1. features + XGBoost            acc=0.6340  macroF1=0.5421  kappa=0.4733
   per-class F1: W=0.730  N1=0.213  N2=0.669  N3=0.523  R=0.576
1b. + HMM decoding               acc=0.6370  macroF1=0.5391  kappa=0.4719
   per-class F1: W=0.730  N1=0.182  N2=0.667  N3=0.508  R=0.608


HMM decoding trades macro-F1 for accuracy: it removes short transient N1 segments, which
helps the majority classes and hurts the rarest one. Both operating points are kept.

## Experiment 2: Raw-signal CNN (per epoch)

Skip hand-designed features and learn filters directly from the waveform. Large first
kernel (`FS//2`) to capture rhythms, then stacked small kernels.

In [10]:
class StagingCNN(nn.Module):
    def __init__(self, in_ch=7, n_classes=5, dropout=0.5):
        super().__init__()
        k1, s1 = FS//2, max(FS//16, 1)
        self.features = nn.Sequential(
            nn.Conv1d(in_ch, 128, k1, stride=s1, padding=k1//2, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(True), nn.MaxPool1d(8,8), nn.Dropout(dropout),
            nn.Conv1d(128,128,8,padding=4,bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Conv1d(128,128,8,padding=4,bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Conv1d(128,128,8,padding=4,bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.MaxPool1d(4,4), nn.Dropout(dropout))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, n_classes))
    def forward(self, x):                       # x [B, 7, 3000]
        return self.head(self.pool(self.features(x)))

m = StagingCNN()
print("output:", tuple(m(torch.randn(2,7,3000)).shape),
      "| params:", f"{sum(p.numel() for p in m.parameters()):,}")

output: (2, 5) | params: 439,685


In [11]:
class EpochDS(Dataset):
    """one 30 s epoch -> one label, z-scored per channel"""
    def __init__(self, subjects):
        self.items = [(s,i) for s in subjects for i in range(len(Yl[s]))]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        s, j = self.items[i]
        x = X[s][j]
        x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6)
        return torch.from_numpy(x), int(Yl[s][j])

def class_weights(subjects):
    c = np.bincount(np.concatenate([Yl[s] for s in subjects]), minlength=5)
    return torch.tensor(c.sum()/(5*np.maximum(c,1)), dtype=torch.float32, device=DEVICE)

print("epochs in fold-0 train set:", len(EpochDS(FOLDS[0][0])))

epochs in fold-0 train set: 11914


In [12]:
def train_cnn(tr, te, epochs=EPOCHS):
    model = StagingCNN().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(weight=class_weights(tr))
    dl    = DataLoader(EpochDS(tr), batch_size=128, shuffle=True, num_workers=0, drop_last=True)

    for ep in range(epochs):
        model.train(); run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); run += loss.item()
        sched.step()
        print(f"    epoch {ep+1:2d}/{epochs}  loss {run/len(dl):.4f}", flush=True)

    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for s in te:
            x = X[s]
            x = (x - x.mean(2, keepdims=True)) / (x.std(2, keepdims=True) + 1e-6)
            pr = []
            for i in range(0, len(x), 256):
                pr.append(model(torch.from_numpy(x[i:i+256]).to(DEVICE)).softmax(-1).cpu().numpy())
            yt.append(Yl[s]); yp.append(np.concatenate(pr).argmax(1))
    return np.concatenate(yt), np.concatenate(yp)

yts, yps = [], []
for fi, (tr, te) in enumerate(FOLDS):
    print(f"  fold {fi}", flush=True)
    a, b = train_cnn(tr, te); yts.append(a); yps.append(b)
RESULTS["2. raw-signal CNN"] = report("2. raw-signal CNN", np.concatenate(yts), np.concatenate(yps))

  fold 0
    epoch  1/10  loss 1.0572
    epoch  2/10  loss 0.7751
    epoch  3/10  loss 0.7090
    epoch  4/10  loss 0.6733
    epoch  5/10  loss 0.6458
    epoch  6/10  loss 0.6171
    epoch  7/10  loss 0.6046
    epoch  8/10  loss 0.5806
    epoch  9/10  loss 0.5578
    epoch 10/10  loss 0.5525
  fold 1
    epoch  1/10  loss 1.0363
    epoch  2/10  loss 0.6849
    epoch  3/10  loss 0.6231
    epoch  4/10  loss 0.5904
    epoch  5/10  loss 0.5558
    epoch  6/10  loss 0.5316
    epoch  7/10  loss 0.5065
    epoch  8/10  loss 0.4828
    epoch  9/10  loss 0.4663
    epoch 10/10  loss 0.4570
  fold 2
    epoch  1/10  loss 0.9886
    epoch  2/10  loss 0.7286
    epoch  3/10  loss 0.6798
    epoch  4/10  loss 0.6456
    epoch  5/10  loss 0.6036
    epoch  6/10  loss 0.5923
    epoch  7/10  loss 0.5703
    epoch  8/10  loss 0.5446
    epoch  9/10  loss 0.5307
    epoch 10/10  loss 0.5171
2. raw-signal CNN                acc=0.6150  macroF1=0.5684  kappa=0.5008
   per-class F1: W=0.803  N1=

## Experiment 3: CNN + BiLSTM over epoch sequences

Sleep stages are strongly autocorrelated, so give the model temporal context: encode each
epoch with the CNN, then run a bidirectional LSTM across a window of consecutive epochs
and label every epoch in the window.

In [13]:
SEQ_LEN = 20

class SeqDS(Dataset):
    def __init__(self, subjects, L=SEQ_LEN, stride=None):
        stride = stride or L//2
        self.idx = [(s,a) for s in subjects
                    for a in range(0, max(1, len(Yl[s])-L+1), stride)]
        self.L = L
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        s, a = self.idx[i]; L = self.L
        x = X[s][a:a+L]; y = Yl[s][a:a+L]
        if len(y) < L:                                   # pad the tail
            pad = L - len(y)
            x = np.concatenate([x, np.zeros((pad,7,3000), np.float32)])
            y = np.concatenate([y, np.zeros(pad, np.int64)])
        x = (x - x.mean(2, keepdims=True)) / (x.std(2, keepdims=True) + 1e-6)
        return torch.from_numpy(x), torch.from_numpy(y)

class StagingSeqNet(nn.Module):
    def __init__(self, in_ch=7, n_classes=5, hidden=128, layers=2, dropout=0.5):
        super().__init__()
        self.features = StagingCNN(in_ch=in_ch, dropout=dropout).features
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.lstm = nn.LSTM(128, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers>1 else 0.)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2*hidden, n_classes))
    def forward(self, x):                                # x [B, L, 7, 3000]
        B, L, C, T = x.shape
        f = self.pool(self.features(x.reshape(B*L, C, T))).flatten(1).reshape(B, L, 128)
        h, _ = self.lstm(f)
        return self.head(h)                              # [B, L, 5]

m = StagingSeqNet()
print("output:", tuple(m(torch.randn(1,4,7,3000)).shape),
      "| params:", f"{sum(p.numel() for p in m.parameters()):,}")

output: (1, 4, 5) | params: 1,099,781


In [14]:
def train_seq(tr, te, epochs=EPOCHS):
    model = StagingSeqNet().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(weight=class_weights(tr))
    dl    = DataLoader(SeqDS(tr), batch_size=8, shuffle=True, num_workers=0)

    for ep in range(epochs):
        model.train(); run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).reshape(-1,5), yb.reshape(-1))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); run += loss.item()
        sched.step()
        print(f"    epoch {ep+1:2d}/{epochs}  loss {run/len(dl):.4f}", flush=True)

    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for s in te:
            x = X[s]; n = len(x); pad = (-n) % SEQ_LEN
            xs = np.concatenate([x, np.zeros((pad,7,3000), np.float32)]) if pad else x
            xs = (xs - xs.mean(2, keepdims=True)) / (xs.std(2, keepdims=True) + 1e-6)
            xs = xs.reshape(-1, SEQ_LEN, 7, 3000)
            out = []
            for i in range(0, len(xs), 4):
                out.append(model(torch.from_numpy(xs[i:i+4]).to(DEVICE))
                           .softmax(-1).cpu().numpy().reshape(-1,5))
            yt.append(Yl[s]); yp.append(np.concatenate(out)[:n].argmax(1))
    return np.concatenate(yt), np.concatenate(yp)

yts, yps = [], []
for fi, (tr, te) in enumerate(FOLDS):
    print(f"  fold {fi}", flush=True)
    a, b = train_seq(tr, te); yts.append(a); yps.append(b)
RESULTS["3. CNN + BiLSTM"] = report("3. CNN + BiLSTM", np.concatenate(yts), np.concatenate(yps))

  fold 0
    epoch  1/10  loss 1.1693
    epoch  2/10  loss 0.8888
    epoch  3/10  loss 0.8355
    epoch  4/10  loss 0.7593
    epoch  5/10  loss 0.7146
    epoch  6/10  loss 0.6827
    epoch  7/10  loss 0.6329
    epoch  8/10  loss 0.5886
    epoch  9/10  loss 0.5605
    epoch 10/10  loss 0.5489
  fold 1
    epoch  1/10  loss 1.1684
    epoch  2/10  loss 0.8623
    epoch  3/10  loss 0.7668
    epoch  4/10  loss 0.6860
    epoch  5/10  loss 0.6439
    epoch  6/10  loss 0.5702
    epoch  7/10  loss 0.5419
    epoch  8/10  loss 0.5074
    epoch  9/10  loss 0.4877
    epoch 10/10  loss 0.4660
  fold 2
    epoch  1/10  loss 1.1704
    epoch  2/10  loss 0.9380
    epoch  3/10  loss 0.8613
    epoch  4/10  loss 0.7610
    epoch  5/10  loss 0.7155
    epoch  6/10  loss 0.6884
    epoch  7/10  loss 0.6315
    epoch  8/10  loss 0.5636
    epoch  9/10  loss 0.5528
    epoch 10/10  loss 0.5566
3. CNN + BiLSTM                  acc=0.6505  macroF1=0.6157  kappa=0.5404
   per-class F1: W=0.811  N1=

## Experiment 4: Hemispheric-asymmetry graph + state-space model

Stroke is lateralized, so the difference between homologous derivations (C4↔C3, O2↔O1)
should carry lesion information that a channel-symmetric model discards.

Encode each channel, attend over a montage graph, pool the **signed difference** of
homologous pairs, then run a selective state-space model across epochs.

In [15]:
HOMOLOGOUS = [(0,1), (2,3)]                       # C4<->C3, O2<->O1
EDGES = [(0,1),(2,3),(0,2),(1,3),(4,5),(0,4),(1,5),(2,4),(3,5),(6,4),(6,5)]

def build_adj(n=7):
    A = torch.eye(n)
    for i,j in EDGES: A[i,j] = 1; A[j,i] = 1
    return A

class ChannelCNN(nn.Module):
    """encode one 30 s channel -> embedding of size d"""
    def __init__(self, d=48, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1,32,50,stride=6,padding=25,bias=False), nn.BatchNorm1d(32), nn.ReLU(True),
            nn.MaxPool1d(8,8), nn.Dropout(dropout),
            nn.Conv1d(32,64,8,padding=4,bias=False), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(4,4), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(64, d)
    def forward(self, x):                          # [B*7, 1, 3000]
        return self.fc(self.net(x).flatten(1))

class GraphAttention(nn.Module):
    """GAT-style attention restricted to the montage graph"""
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d,d,bias=False); self.a = nn.Linear(2*d,1,bias=False)
        self.register_buffer("adj", build_adj()); self.leaky = nn.LeakyReLU(0.2)
    def forward(self, h):                          # [B,7,d]
        B,C,d = h.shape; Wh = self.W(h)
        e = self.leaky(self.a(torch.cat([Wh.unsqueeze(2).expand(B,C,C,d),
                                         Wh.unsqueeze(1).expand(B,C,C,d)], -1)).squeeze(-1))
        e = e.masked_fill(~(self.adj>0).unsqueeze(0), float("-inf"))
        return torch.nn.functional.elu(torch.einsum("bij,bjd->bid", torch.softmax(e,-1), Wh)) + h

class AsymPool(nn.Module):
    """the clinical prior: signed difference between homologous derivations"""
    def forward(self, h):                          # [B,7,d] -> [B,2d]
        return torch.cat([(h[:,i]-h[:,j]).abs() for i,j in HOMOLOGOUS], -1)

print("graph edges:", len(EDGES), "| homologous pairs:", HOMOLOGOUS)

graph edges: 11 | homologous pairs: [(0, 1), (2, 3)]


In [16]:
class SelectiveSSM(nn.Module):
    """Mamba-style selective state-space scan over the epoch sequence"""
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.di = expand*d_model; self.dt_rank = max(8, d_model//16); self.ds = d_state
        self.in_proj  = nn.Linear(d_model, 2*self.di)
        self.x_proj   = nn.Linear(self.di, self.dt_rank + 2*d_state)
        self.dt_proj  = nn.Linear(self.dt_rank, self.di)
        self.A_log    = nn.Parameter(torch.log(torch.arange(1, d_state+1, dtype=torch.float32)
                                               .repeat(self.di,1)))
        self.D        = nn.Parameter(torch.ones(self.di))
        self.out_proj = nn.Linear(self.di, d_model)
    def forward(self, x):                          # [B,L,d]
        B,L,_ = x.shape
        xi, z = self.in_proj(x).chunk(2, -1)
        xi = torch.nn.functional.silu(xi)
        A  = -torch.exp(self.A_log.clamp(-6,6))                    # bounded decay
        dt, Bm, Cm = torch.split(self.x_proj(xi), [self.dt_rank, self.ds, self.ds], -1)
        dt = torch.nn.functional.softplus(self.dt_proj(dt)).clamp(max=6.0)
        h, ys = x.new_zeros(B, self.di, self.ds), []
        for t in range(L):                                          # sequential scan
            h = torch.exp(dt[:,t].unsqueeze(-1)*A)*h \
                + (dt[:,t].unsqueeze(-1)*Bm[:,t].unsqueeze(1))*xi[:,t].unsqueeze(-1)
            ys.append(torch.einsum("bds,bs->bd", h, Cm[:,t]))
        return self.out_proj((torch.stack(ys,1) + self.D*xi) * torch.nn.functional.silu(z))

class AsymGraphSSM(nn.Module):
    def __init__(self, d=48, D=128, n_classes=5, dropout=0.3):
        super().__init__()
        self.enc  = ChannelCNN(d, dropout)
        self.gat  = GraphAttention(d)
        self.asym = AsymPool()
        self.fuse = nn.Sequential(nn.Linear(3*d, D), nn.GELU(), nn.Dropout(dropout))
        self.ssm  = SelectiveSSM(D)
        self.norm = nn.LayerNorm(D)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(D, n_classes))
    def forward(self, x):                          # [B,L,7,3000]
        B,L,C,T = x.shape
        h = self.enc(x.reshape(B*L*C, 1, T)).reshape(B*L, C, -1)   # [B*L,7,d]
        h = self.gat(h)
        e = self.fuse(torch.cat([h.mean(1), self.asym(h)], -1)).reshape(B, L, -1)
        return self.head(self.norm(e + self.ssm(e)))               # [B,L,5]

m = AsymGraphSSM()
print("output:", tuple(m(torch.randn(1,4,7,3000)).shape),
      "| params:", f"{sum(p.numel() for p in m.parameters()):,}")

output: (1, 4, 5) | params: 159,037


In [17]:
def train_asym(tr, te, epochs=EPOCHS):
    model = AsymGraphSSM().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=3e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(weight=class_weights(tr))
    dl    = DataLoader(SeqDS(tr, L=SEQ_LEN), batch_size=4, shuffle=True, num_workers=0)

    for ep in range(epochs):
        model.train(); run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).reshape(-1,5), yb.reshape(-1))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); run += loss.item()
        sched.step()
        print(f"    epoch {ep+1:2d}/{epochs}  loss {run/len(dl):.4f}", flush=True)

    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for s in te:
            x = X[s]; n = len(x); pad = (-n) % SEQ_LEN
            xs = np.concatenate([x, np.zeros((pad,7,3000), np.float32)]) if pad else x
            xs = (xs - xs.mean(2, keepdims=True)) / (xs.std(2, keepdims=True) + 1e-6)
            xs = xs.reshape(-1, SEQ_LEN, 7, 3000)
            out = []
            for i in range(0, len(xs), 2):
                out.append(model(torch.from_numpy(xs[i:i+2]).to(DEVICE))
                           .softmax(-1).cpu().numpy().reshape(-1,5))
            yt.append(Yl[s]); yp.append(np.concatenate(out)[:n].argmax(1))
    return np.concatenate(yt), np.concatenate(yp)

yts, yps = [], []
for fi, (tr, te) in enumerate(FOLDS):
    print(f"  fold {fi}", flush=True)
    a, b = train_asym(tr, te); yts.append(a); yps.append(b)
RESULTS["4. asymmetry graph + SSM"] = report("4. asymmetry graph + SSM",
                                             np.concatenate(yts), np.concatenate(yps))

  fold 0
    epoch  1/10  loss 1.3939
    epoch  2/10  loss 1.2328
    epoch  3/10  loss 1.1302
    epoch  4/10  loss 1.0936
    epoch  5/10  loss 1.0651
    epoch  6/10  loss 0.9671
    epoch  7/10  loss 0.8805
    epoch  8/10  loss 0.8138
    epoch  9/10  loss 0.7634
    epoch 10/10  loss 0.7404
  fold 1
    epoch  1/10  loss 1.3362
    epoch  2/10  loss 1.1143
    epoch  3/10  loss 1.0341
    epoch  4/10  loss 0.9274
    epoch  5/10  loss 0.8810
    epoch  6/10  loss 0.7889
    epoch  7/10  loss 0.7385
    epoch  8/10  loss 0.6780
    epoch  9/10  loss 0.6211
    epoch 10/10  loss 0.6161
  fold 2
    epoch  1/10  loss 1.4204
    epoch  2/10  loss 1.1836
    epoch  3/10  loss 1.1090
    epoch  4/10  loss 1.0063
    epoch  5/10  loss 0.9084
    epoch  6/10  loss 0.8549
    epoch  7/10  loss 0.7894
    epoch  8/10  loss 0.7380
    epoch  9/10  loss 0.7103
    epoch 10/10  loss 0.6868
4. asymmetry graph + SSM         acc=0.5649  macroF1=0.5137  kappa=0.4274
   per-class F1: W=0.705  N1=